# AKT 2 — interpretacja osi: czym jest to, co model odkrył?

## Cel

Akt 1 znalazł oś `v_clean` rozdzielającą chosen/rejected, która przetrwała neutralizację konfoundów i przeszła test spójności (gradient per sim_level). Teraz pytamy: **czym ona jest w języku pojęć?**

## Metoda — sondy tekstowe CLIP z trzema zabezpieczeniami

1. **Sondy jako pary kontrastowe** `text(pozytyw) − text(negatyw)` — niweluje modality gap (odejmowanie dwóch tekstów kasuje wspólny "stożek tekstowy")
2. **Baseline losowy** — w 512D losowe kierunki mają |cos| ≈ 1/√512 ≈ 0.044; wszystko wyraźnie powyżej jest nieprzypadkowe
3. **Sanity check na osiach konfoundów** — te same sondy na `v_sharp` (oś ostrości z Laplasjanu): jeśli kategoria TECHNICZNA wygra dla v_sharp, metoda jest skalibrowana. Pozytywna kontrola.

Plus **ekstremy osi** — top/bottom zdjęcia wg rzutu: ludzkie oko jako ostateczny interpretator.

## Cztery kategorie sond (możliwe wyniki)

| Wygrywa | Znaczenie |
|---|---|
| CZASOWE | model odkrył moment czasowy (sukces marzeń) |
| KOMPOZYCYJNE | moment kompozycyjny (dobry, uczciwy wynik) |
| EKSPRESYJNE | "twarze, które mówią" (blisko Strykera) |
| TECHNICZNE | resztka jakości mimo neutralizacji (sygnał ostrzegawczy) |


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q open_clip_torch 2>/dev/null


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, open_clip
np.set_printoptions(precision=4, suppress=True)

DRIVE = Path('/content/drive/MyDrive/fsa_data')
AXES = DRIVE / 'akt1_axes.npz'
PAIRS = DRIVE / 'fsa_pairs_filtered.csv'
EMB_CACHE = DRIVE / 'clip_embeddings.npz'
IMAGES_DIR = DRIVE / 'fsa_images'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model = model.to(device).eval()
print(f'✅ CLIP (obraz+tekst) na {device}')


## 1. Wczytaj osie z Aktu 1 + embeddingi

In [ ]:
with np.load(AXES) as d:
    v_clean = d['v_clean']    # oś czysta (mean-diff po neutralizacji)
    w_clean = d['w_clean']    # oś probe po neutralizacji
    v_raw   = d['v_raw']      # oś surowa (do porównania)
    v_hole  = d['v_hole']     # kierunek dziurki (kontrola)
    v_sharp = d['v_sharp']    # kierunek ostrości z Laplasjanu (POZYTYWNA KONTROLA)
print(f'Osie wczytane, dim={len(v_clean)}')
print(f'cos(v_clean, v_raw)   = {np.dot(v_clean, v_raw):.3f}  (jak bardzo neutralizacja zmieniła oś)')
print(f'cos(v_clean, w_clean) = {np.dot(v_clean, w_clean):.3f}  (zgodność mean-diff i probe)')

pairs = pd.read_csv(PAIRS, dtype={'similarity': float}, keep_default_na=False)
with np.load(EMB_CACHE, allow_pickle=True) as data:
    all_files = list(data['files'])
    all_emb = data['emb']
file_to_idx = {f: i for i, f in enumerate(all_files)}
print(f'Pary: {len(pairs)}, embeddingi: {all_emb.shape}')


## 2. Sondy — pary kontrastowe w czterech kategoriach

Każda sonda = (pozytyw, negatyw). Kierunek sondy = text(poz) − text(neg), znormalizowany. Kontrast niweluje modality gap.


In [ ]:
PROBES = {
  'CZASOWE': [
    ("a photograph capturing the decisive moment", "a photograph of an ordinary moment"),
    ("a person caught mid-gesture",                "a person standing still"),
    ("the peak of the action",                     "before the action begins"),
    ("a fleeting spontaneous instant",             "a static posed scene"),
    ("candid movement frozen in time",             "a motionless posed subject"),
  ],
  'KOMPOZYCYJNE': [
    ("a perfectly composed photograph",                    "a poorly composed photograph"),
    ("balanced framing with strong visual structure",      "awkward framing with cluttered structure"),
    ("an elegant geometric composition",                   "a chaotic random arrangement"),
    ("a well-framed subject with clear focal point",       "a badly cropped subject with no focal point"),
    ("harmonious visual balance in the frame",             "unbalanced distracting composition"),
  ],
  'TECHNICZNE': [
    ("a sharp well-exposed photograph",            "a blurry poorly exposed photograph"),
    ("a high quality professional photograph",     "a low quality amateur snapshot"),
    ("a clean undamaged photograph",               "a damaged photograph with marks and scratches"),
    ("a crisp detailed image",                     "a soft out-of-focus image"),
  ],
  'EKSPRESYJNE': [
    ("a powerful emotional expression",       "a blank neutral expression"),
    ("an intimate human connection",          "a distant detached scene"),
    ("a compelling human story",              "an uneventful empty scene"),
    ("an expressive face full of feeling",    "an expressionless face"),
    ("people interacting with each other",    "people ignoring each other"),
  ],
}

@torch.no_grad()
def text_embed(prompt):
    t = model.encode_text(tokenizer([prompt]).to(device))
    t = t / t.norm(dim=-1, keepdim=True)
    return t[0].cpu().numpy().astype(np.float64)

probe_dirs = {}   # (kategoria, i) -> (nazwa, kierunek)
for cat, plist in PROBES.items():
    for i, (pos, neg) in enumerate(plist):
        v = text_embed(pos) - text_embed(neg)
        v = v / (np.linalg.norm(v) + 1e-12)
        probe_dirs[(cat, i)] = (f'{pos[:42]}...', v)
print(f'Zbudowano {len(probe_dirs)} sond kontrastowych w {len(PROBES)} kategoriach')


## 3. Baseline losowy — ile cosinusa daje przypadek w 512D

In [ ]:
rng = np.random.RandomState(0)
rand_cos = []
for _ in range(2000):
    r = rng.randn(len(v_clean)); r /= np.linalg.norm(r)
    rand_cos.append(abs(np.dot(v_clean, r)))
rand_cos = np.array(rand_cos)
thr95, thr99 = np.percentile(rand_cos, [95, 99])
print(f'Losowe kierunki: |cos| średnio {rand_cos.mean():.4f} (teoria 1/√512={1/np.sqrt(512):.4f})')
print(f'Próg istotności: 95%={thr95:.4f}, 99%={thr99:.4f}')
print('→ sonda z |cos| powyżej progu 99% jest nieprzypadkowo powiązana z osią')


## 4. GŁÓWNY POMIAR — cosinusy sond z osią czystą

In [ ]:
def probe_axis(axis, axis_name):
    rows = []
    for (cat, i), (name, pv) in probe_dirs.items():
        c = float(np.dot(axis, pv))
        rows.append({'kategoria': cat, 'sonda': name, 'cos': c})
    df = pd.DataFrame(rows)
    print(f'=== {axis_name} ===')
    print('\nRanking sond (|cos| malejąco):')
    dfs = df.reindex(df['cos'].abs().sort_values(ascending=False).index)
    for _, r in dfs.head(10).iterrows():
        sig = '**' if abs(r['cos']) > thr99 else ('*' if abs(r['cos']) > thr95 else '  ')
        print(f"  {sig} {r['cos']:+.4f}  [{r['kategoria'][:5]}] {r['sonda']}")
    print(f"\n  (** = powyżej progu 99% losowości {thr99:.3f}, * = powyżej 95% {thr95:.3f})")
    agg = df.groupby('kategoria')['cos'].agg(['mean', lambda x: x.abs().mean()])
    agg.columns = ['średni cos (ze znakiem)', 'średni |cos|']
    agg = agg.sort_values('średni |cos|', ascending=False)
    print('\nAgregacja per kategoria:')
    print(agg.to_string())
    return df, agg

df_clean, agg_clean = probe_axis(v_clean, 'OŚ CZYSTA (kandydat na oś momentu)')


## 5. POZYTYWNA KONTROLA — te same sondy na osi ostrości

Kalibracja metody: `v_sharp` pochodzi z twardych etykiet Laplasjanu, więc WIEMY, czym jest. Jeśli sondy poprawnie wskażą kategorię TECHNICZNĄ — metoda działa i możemy ufać wynikowi dla osi czystej.


In [ ]:
df_sharp, agg_sharp = probe_axis(v_sharp, 'OŚ OSTROŚCI (pozytywna kontrola — powinna wygrać TECHNICZNE)')
print()
df_hole, agg_hole = probe_axis(v_hole, 'OŚ DZIURKI (kontrola nr 2 — oczekiwane: damaged/marks w TECHNICZNE)')

winner_sharp = agg_sharp.index[0]
print('\n' + '='*60)
if winner_sharp == 'TECHNICZNE':
    print('✅ KALIBRACJA OK: sondy poprawnie nazwały oś ostrości jako TECHNICZNĄ.')
    print('   Metodzie można ufać przy interpretacji osi czystej.')
else:
    print(f'⚠ KALIBRACJA WĄTPLIWA: dla osi ostrości wygrała kategoria {winner_sharp}.')
    print('   Wyniki dla osi czystej traktuj ostrożnie — sondy mogą nie działać.')


## 6. Wykres porównawczy — profil osi czystej vs kontrolnych

In [ ]:
cats = list(PROBES.keys())
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(cats)); w = 0.27
for off, (agg, label, col) in zip([-w, 0, w], [
        (agg_clean, 'oś CZYSTA (moment?)', 'darkgreen'),
        (agg_sharp, 'oś ostrości (kontrola)', 'orange'),
        (agg_hole,  'oś dziurki (kontrola)', 'gray')]):
    vals = [agg.loc[c, 'średni |cos|'] if c in agg.index else 0 for c in cats]
    ax.bar(x + off, vals, w, label=label, color=col, alpha=0.85)
ax.axhline(thr99, color='red', ls='--', lw=1, label=f'próg 99% losowości ({thr99:.3f})')
ax.set_xticks(x); ax.set_xticklabels(cats)
ax.set_ylabel('średni |cos| kategorii z osią')
ax.set_title('Profil pojęciowy osi — która kategoria sond najlepiej opisuje każdą oś', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


## 7. EKSTREMA OSI — oczy jako ostateczny interpretator

Rzutujemy wszystkie obrazy par na oś czystą i pokazujemy skrajności: najbardziej "chosen-like" (wysoki rzut) i "rejected-like" (niski). Szukaj wzorca, którego sondy mogły nie nazwać.


In [ ]:
# Rzut wszystkich unikalnych obrazów z par na oś czystą
used = pd.unique(pairs[['chosen_file','rejected_file']].values.ravel())
idxs = np.array([file_to_idx[f] for f in used if f in file_to_idx])
names = np.array([f for f in used if f in file_to_idx])
proj = all_emb[idxs].astype(np.float64) @ v_clean

order = np.argsort(proj)
N = 8
bottom = names[order[:N]]     # najbardziej 'rejected-like'
top = names[order[-N:]][::-1] # najbardziej 'chosen-like'

fig, axes = plt.subplots(2, N, figsize=(2.2*N, 5.5))
for j, f in enumerate(top):
    axes[0, j].imshow(Image.open(IMAGES_DIR / f)); axes[0, j].axis('off')
    axes[0, j].set_title(f'+{proj[order[-1-j]]:.2f}', fontsize=8, color='green')
for j, f in enumerate(bottom):
    axes[1, j].imshow(Image.open(IMAGES_DIR / f)); axes[1, j].axis('off')
    axes[1, j].set_title(f'{proj[order[j]]:.2f}', fontsize=8, color='red')
axes[0, 0].set_ylabel('MAX oś', fontsize=10)
axes[1, 0].set_ylabel('MIN oś', fontsize=10)
plt.suptitle('Ekstremy osi czystej — góra: najbardziej chosen-like, dół: rejected-like', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('Pytania do oka: co łączy górny rząd? co łączy dolny? gest/ruch? kompozycja? treść?')


## 8. Ekstremalne PARY — gdzie oś najmocniej rozdziela chosen od rejected

In [ ]:
# Delta rzutu wewnątrz pary: p_chosen - p_rejected (analiza parowana)
ci = pairs['chosen_file'].map(lambda f: file_to_idx.get(f,-1)).values
ri = pairs['rejected_file'].map(lambda f: file_to_idx.get(f,-1)).values
ok = (ci>=0)&(ri>=0)
p_c = all_emb[ci[ok]].astype(np.float64) @ v_clean
p_r = all_emb[ri[ok]].astype(np.float64) @ v_clean
delta = p_c - p_r
pairs_ok = pairs[ok].reset_index(drop=True)

print(f'Analiza parowana (bonus z notatki o d\'):')
print(f'  frakcja par gdzie chosen > rejected wzdłuż osi: {np.mean(delta>0):.2%}')
print(f'  mediana Δ: {np.median(delta):+.4f}')

# Pokaż 4 pary o największej delcie — tam oś 'widzi' różnicę najmocniej
top_pairs = np.argsort(delta)[::-1][:4]
fig, axes = plt.subplots(4, 2, figsize=(8, 15))
for row, pi in enumerate(top_pairs):
    r = pairs_ok.iloc[pi]
    axes[row,0].imshow(Image.open(IMAGES_DIR / r['chosen_file'])); axes[row,0].axis('off')
    axes[row,0].set_title(f'CHOSEN  p={p_c[pi]:+.2f}', fontsize=9, color='green')
    axes[row,1].imshow(Image.open(IMAGES_DIR / r['rejected_file'])); axes[row,1].axis('off')
    axes[row,1].set_title(f'REJECTED  p={p_r[pi]:+.2f}  (Δ={delta[pi]:.2f})', fontsize=9, color='red')
plt.suptitle('Pary o największej różnicy wzdłuż osi — co oś "widzi"?', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 9. Zapis wyników + werdykt

In [ ]:
winner = agg_clean.index[0]
second = agg_clean.index[1]
margin = agg_clean.iloc[0]['średni |cos|'] - agg_clean.iloc[1]['średni |cos|']

print('='*60)
print('WERDYKT INTERPRETACYJNY (Akt 2)')
print('='*60)
print(f'Kalibracja (oś ostrości → TECHNICZNE?): {"TAK ✅" if agg_sharp.index[0]=="TECHNICZNE" else "NIE ⚠"}')
print(f'Oś czysta — wygrywa kategoria: {winner} (nad {second}, margines {margin:.4f})')
print(f'Sondy powyżej progu 99%: {int((df_clean.cos.abs() > thr99).sum())}/{len(df_clean)}')
print()
print('Pamiętaj: sondy są SUGESTYWNE, nie dowodowe (modality gap).')
print('Ostateczna interpretacja = sondy + kalibracja + ekstremy (oczy) razem.')

out = {
    'winner': winner, 'margin': float(margin),
    'calibration_ok': bool(agg_sharp.index[0]=='TECHNICZNE'),
    'agg_clean': {c: float(agg_clean.loc[c,'średni |cos|']) for c in agg_clean.index},
    'agg_sharp': {c: float(agg_sharp.loc[c,'średni |cos|']) for c in agg_sharp.index},
    'thr95': float(thr95), 'thr99': float(thr99),
    'paired_fraction': float(np.mean(delta>0)),
    'probes': df_clean.to_dict('records'),
}
with open(DRIVE / 'akt2_results.json', 'w') as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print(f'\n✅ Zapisano akt2_results.json')
